In [ ]:
# 加载环境变量
from dotenv import load_dotenv

load_dotenv()

# 1.没有记忆时

In [ ]:
from langchain.agents import create_agent

agent = create_agent(model = "deepseek-chat")


In [ ]:
from langchain.messages import HumanMessage

# 第一次调用，告知AI我的信息
response = agent.invoke(
    {"messages": [HumanMessage(content="你好，我叫虎哥，我最喜欢猫猫。")]}
)

print(response)

In [ ]:
# 第二次调用，询问我的信息
response = agent.invoke(
    {"messages": [HumanMessage(content="我最喜欢什么动物?")]}
)

print(response)

# 2.添加记忆

Agent的记忆分成两类
- 短期记忆：当前任务或会话的上下文
    - 在Langchain中默认是使用AgentState来实现的，使用会话历史来记录之前的访问信息。
    - 基本原来就是每次与大模型询问之后会产生一个快照。然后在询问大模型的时候带上最近的快照。
        - 可以回到某一次快照来实现时间穿梭
- 长期记忆：跨任务/会话的经验与知识

基于Langchain文档的信息：https://docs.langchain.com/oss/python/langchain/short-term-memory#short-term-memory

添加会话记忆（短期记忆）分为三步：
- 导入并初始化Checkpointer
- 创建Agent，指定Checkpointer
- 调用Agent，指定thread_id




In [ ]:
#下面代码实际来源于官方网站提供的样例
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver# 保存在内存中，生产环境中建议是使用数据库存储

agent = create_agent(
    "deepseek-chat",
    checkpointer=InMemorySaver()# 使用内存保存检查点
)

In [ ]:
from langchain.messages import HumanMessage

# 设定thread_id，作为会话标识
config = {"configurable": {"thread_id": "thread_1"}}

# 第一次调用，告知AI我的信息
response = agent.invoke(
    {"messages": [HumanMessage(content="你好，我叫虎哥，我最喜欢猫猫。")]},
    config # 调用时添加thread_id->也就是上面我们设置的会话标识
)

print(response)

In [ ]:
# 第二次调用，询问我的信息，这次带上thread_id，唤起记忆
response = agent.invoke(
    {"messages": [HumanMessage(content="我最喜欢的动物是什么？")]},
    config # 调用时添加thread_id 调用我们上一次问答时候的快照&回答相关消息
)

print(response)

# 3.Memory持久化存储

上面我们是将记忆存储在内存中，但是在实际的生产环境中，我们一般是将记忆存储在数据库中。

这里我们选择使用Sqlite作为存储方案，首先需要按照langgraph-checkpoint-sqlite依赖：
```
uv add langgraph-checkpoint-sqlite
```
接着，按照以下步骤使用：
- 导入依赖
- 初始化checkpointer
- 自动建表
- 创建Agent，指定checkpointer

这里使用sqlite是因为数据被写入特定的一个.db文件，所有说只要读取同一个.db文件就能实现数据库的管理
但是我们实际开发中是要和数据库进行连接的，正常使用的是PostgreSQL。配置详情看官方文档。


In [ ]:
import sqlite3
from langgraph.checkpoint.sqlite import SqliteSaver

# 连接sqlite
connection = sqlite3.connect("resources/checkpoint.db", check_same_thread=False)#指定数据库文件和位置
# 初始化checkpointer
checkpointer = SqliteSaver(connection)#将连接的对象传给checkpointer
# 自动建表
checkpointer.setup()

# 创建agent
agent = create_agent(
    "deepseek-chat",
    checkpointer=checkpointer,
)

In [ ]:
from langchain.messages import HumanMessage

# 设定thread_id，作为会话标识
config = {"configurable": {"thread_id": "thread_2"}}#这个id可以自己指定，方便自己后面的管理

# 第一次调用，告知AI我的信息
response = agent.invoke(
    {"messages": [HumanMessage(content="你好，我叫虎哥，我最喜欢猫猫。")]},
    config # 调用时添加thread_id
)

print(response)

In [ ]:
# 第二次调用，询问我的信息，这次带上thread_id，唤起记忆
response = agent.invoke(
    {"messages": [HumanMessage(content="我最喜欢的动物是什么？")]},
    config # 调用时添加thread_id
)

print(response)

# 4.记忆管理

当会话历史过长时，可能会超出模型的上下文窗口限制，常见的解决方案有：
- 修剪消息
- 删除消息
- 总结消息摘要

这里我们演示总结消息摘要的方案


In [ ]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware #我们是引入一个中间键的方法来处理
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.runnables import RunnableConfig

# 初始化checkpointer
checkpointer = InMemorySaver()
# 初始化中间件
middleware = SummarizationMiddleware(# 设置触发条件n
    model="deepseek-chat",
    trigger=("messages", 6), #  触发时机，当消息数超过3时，进行总结
    keep=("messages", 1) #  保留的会话数，超过2条
)
# 创建agent
agent = create_agent(
    model="deepseek-chat",
    middleware=[middleware],
    checkpointer=checkpointer,
)

config: RunnableConfig = {"configurable": {"thread_id": "thread_3"}}# 设定thread_id，作为会话标识
# 制造长会话历史，连续注入三条消息，此时已经到了我们设定的值，触发中间件进行总结
agent.invoke({"messages": [HumanMessage(content="你好，我是虎哥.")]}, config)
agent.invoke({"messages": [HumanMessage(content="我最喜欢的运动是乒乓")]}, config)
agent.invoke({"messages": [HumanMessage(content="我最喜欢的动物是猫猫")]}, config)
# 测试效果
final_response = agent.invoke({"messages": HumanMessage(content="你还记得我吗？")}, config)

In [ ]:
print(final_response)

In [ ]:

for message in final_response["messages"]:
    message.pretty_print()